In [ ]:
from pathlib import Path

import numpy as np

DATA = Path("../data")
np.set_printoptions(precision=3, suppress=True)

In [ ]:
NUMERIC_COLUMNS = ["seats_purchased", "active_users", "feature_adoption_score", "support_tickets"]

with open(DATA/"monthly_usage.csv") as file:
    header = file.readline().strip().split(",")

usage = np.loadtxt(
    DATA/"monthly_usage.csv",
    delimiter=",",
    skiprows=1,
    usecols=[header.index(column) for column in NUMERIC_COLUMNS],
)

subscription_ids = np.loadtxt(
    DATA/"monthly_usage.csv",
    delimiter=",",
    skiprows=1,
    usecols=header.index("subscription_id"),
    dtype=str,
)

for name, array in [("usage", usage), ("subscription_ids", subscription_ids)]:
    print(f"{name}")
    print(f"  shape : {array.shape}")
    print(f"  ndim  : {array.ndim}")
    print(f"  dtype : {array.dtype}")
    print(f"  size  : {array.size}")

print(f"\nColumns : {NUMERIC_COLUMNS}")
print("First 3 rows :")
print(usage[:3])
print(subscription_ids[:3])

In [ ]:
MONTHS = 12

active_users = usage[:, NUMERIC_COLUMNS.index("active_users")].reshape(-1, MONTHS)
support_tickets = usage[:, NUMERIC_COLUMNS.index("support_tickets")].reshape(-1, MONTHS)

seats = usage[::MONTHS, NUMERIC_COLUMNS.index("seats_purchased")]
sub_ids = subscription_ids[::MONTHS]

seats_by_month = usage[:, NUMERIC_COLUMNS.index("seats_purchased")].reshape(-1, MONTHS)
seats_constant = np.all(seats_by_month == seats[:, np.newaxis])

print(f"active_users    : {active_users.shape}")
print(f"support_tickets : {support_tickets.shape}")
print(f"seats           : {seats.shape}")
print(f"sub_ids         : {sub_ids.shape}")
print(f"\nSeats constant within each subscription : {seats_constant}")

print(f"\n{sub_ids[0]} seats : {seats[0]}")
print(f"active_users    : {active_users[0]}")
print(f"support_tickets : {support_tickets[0]}")

In [ ]:
print(f"First subscription, all 12 months ({sub_ids[0]}) :")
print(active_users[0])

print(f"\nLast 3 months of the first two subscriptions {sub_ids[:2]} :")
print(active_users[:2, -3:])

print(f"\nMonth 6 of subscription index 10 ({sub_ids[10]}) :")
print(active_users[10, 5])

picks = [0, 10, 100]

print(f"\nMonth 1 for subscriptions {picks} {sub_ids[picks]} :")
print(active_users[picks, 0])

In [ ]:
utilization = active_users / seats[:, np.newaxis]

fleet_by_month = utilization.mean(axis=0)
per_subscription = utilization.mean(axis=1)

print(f"Fleet average utilization per month (axis=0) {fleet_by_month.shape} :")
print(fleet_by_month)

print(f"\nPer subscription average utilization (axis=1) {per_subscription.shape} :")
print(f"  min  : {per_subscription.min():.3f}")
print(f"  max  : {per_subscription.max():.3f}")
print(f"  mean : {per_subscription.mean():.3f}")

busiest = active_users[0].argmax()

print(f"\nBusiest month for subscription 0 ({sub_ids[0]}) : month {busiest + 1} with {active_users[0, busiest]:.0f} active users")
print(f"Total tickets for subscription 0 : {support_tickets[0].sum():.0f}")
print(f"Total tickets across the fleet : {support_tickets.sum():.0f}")

In [ ]:
recent_utilization = utilization[:, -3:].mean(axis=1)
low_utilization = recent_utilization < 0.5

print(f"Average utilization over the last 3 months {recent_utilization.shape} :")
print(recent_utilization[:5])

print(f"\nMask {low_utilization.shape} {low_utilization.dtype} :")
print(low_utilization[:5])

print(f"\nSubscriptions below 50% : {low_utilization.sum()} of {low_utilization.size} ({low_utilization.mean():.1%})")
print(f"First 5 IDs : {sub_ids[low_utilization][:5]}")
print(f"Their utilization : {recent_utilization[low_utilization][:5]}")